# Caculate F1 score

In [1]:
from sklearn.metrics import f1_score, accuracy_score
import pickle
import numpy as np

def evaluate_by_batch(loaded_results, batch_size=None):
    attributes = list(loaded_results['predictions'].keys())
    results_summary = {}

    for attr in attributes:
        preds = loaded_results['predictions'][attr].squeeze()
        targets = loaded_results['targets'][attr].squeeze()

        binary_preds = (preds >= 0.5).astype(int)
        binary_targets = targets.astype(int)

        if batch_size is None:
            f1 = f1_score(binary_targets, binary_preds)
            acc = accuracy_score(binary_targets, binary_preds)
            results_summary[attr] = {
                'F1-score': round(f1, 4),
                'Accuracy': round(acc, 4)
            }
        else:
            f1_list, acc_list = [], []
            for i in range(0, len(binary_targets), batch_size):
                batch_preds = binary_preds[i:i+batch_size]
                batch_targets = binary_targets[i:i+batch_size]

                if len(np.unique(batch_targets)) > 1:
                    f1 = f1_score(batch_targets, batch_preds)
                else:
                    f1 = 1.0 if (batch_targets == batch_preds).all() else 0.0

                acc = accuracy_score(batch_targets, batch_preds)
                f1_list.append(f1)
                acc_list.append(acc)

            f1_mean, f1_std = np.mean(f1_list), np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)

            results_summary[attr] = {
                'F1-score': round(f1_mean, 4),
                'F1-std': round(f1_std, 4),
                'Accuracy': round(acc_mean, 4),
                'Acc-std': round(acc_std, 4)
            }

    return results_summary

# === 加载结果文件 ===

do_file = 'Bald'
with open(os.path.join(root_dir, do_file,'final_results.pkl'),"rb") as f:
    loaded_results = pickle.load(f)

# === 设置 batch_size（None 为全数据评估）===
batch_size = 256  # 可设为 None

# === 评估 ===
results = evaluate_by_batch(loaded_results, batch_size=batch_size)

# === 打印结果 ===
print("do({}) with".format(do_file) + (f" batch_size={batch_size}" if batch_size else " full-dataset evaluation"))
for attr, scores in results.items():
    if batch_size is None:
        print(f"{attr}: F1 = {scores['F1-score']}, Acc = {scores['Accuracy']}")
    else:
        print(f"{attr}: F1 = {scores['F1-score']} ± {scores['F1-std']}, Acc = {scores['Accuracy']} ± {scores['Acc-std']}")

NameError: name 'os' is not defined

In [27]:
# Find indices of correctly classified samples
attr = 'Bald'
preds = loaded_results['predictions'][attr].squeeze()
targets = loaded_results['targets'][attr].squeeze()

# Convert probabilities/logits to binary predictions
binary_preds = (preds >= 0.5)
binary_targets = targets

indices = loaded_results['indices']

# Boolean mask where prediction equals target
correct_mask = binary_preds == binary_targets

# Boolean mask for male == 0
male_mask = loaded_results['targets']['Male'].squeeze() == 0

# Combine both conditions
final_mask = correct_mask & male_mask

# Select indices that meet both conditions
final_indices = indices[final_mask]
final_indices[:20]


array([  0,  16,  17,  19,  41,  60,  67,  76,  84,  87,  92, 113, 116,
       118, 124, 125, 129, 145, 147, 210])

In [25]:
binary_targets[22]

0.0

In [4]:
from sklearn.metrics import f1_score, accuracy_score
import pickle
import numpy as np
import os

def evaluate_by_batch(loaded_results, batch_size=None):
    attributes = list(loaded_results['predictions'].keys())
    results_summary = {}

    for attr in attributes:
        preds = loaded_results['predictions'][attr].squeeze()
        targets = loaded_results['targets'][attr].squeeze()
        
        binary_preds = (preds >= 0.5).astype(int)
        binary_targets = targets.astype(int)

        if batch_size is None:
            f1 = f1_score(binary_targets, binary_preds)
            acc = accuracy_score(binary_targets, binary_preds)
            results_summary[attr] = {
                'F1-score': round(f1, 3),
                'Accuracy': round(acc, 3)
            }
        else:
            f1_list, acc_list = [], []
            for i in range(0, len(binary_targets), batch_size):
                batch_preds = binary_preds[i:i+batch_size]
                batch_targets = binary_targets[i:i+batch_size]

                if len(np.unique(batch_targets)) > 1:
                    f1 = f1_score(batch_targets, batch_preds)
                else:
                    f1 = 1.0 if (batch_targets == batch_preds).all() else 0.0

                acc = accuracy_score(batch_targets, batch_preds)
                f1_list.append(f1)
                acc_list.append(acc)

            f1_mean, f1_std = np.mean(f1_list), np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)

            results_summary[attr] = {
                'F1-score': round(f1_mean, 3),
                'F1-std': round(f1_std, 2),
                'Accuracy': round(acc_mean, 3),
                'Acc-std': round(acc_std, 2)
            }

    return results_summary


# === 设置评估目标属性 ===
target_attr = "Bald"
batch_size = 256

# === 根目录 ===
#root_dir = "../saved_ablation/celeA_complex/DSCM_effectiveness_step100.0_scale2.5_NTIFalse"
root_dir = "../saved_benchmark/celeA_complex_valid/DSCM_effectiveness_step100.0_scale3.0_NTIFalse_2025-04-23T17-05-57-controlnet_textcond_constrastivegeneration_text_global_after"
# === 遍历所有 do(attr) 文件夹 ===
do_folders = [name for name in ['Young','Male','No_Beard','Bald'] if os.path.isdir(os.path.join(root_dir, name))]


for target_attr in ['Young','Male','No_Beard','Bald']:
    print(f"Comparing '{batch_size}' F1-score on attribute '{target_attr}' under different do(.) conditions:")
    for do_attr in do_folders:
        result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
        if not os.path.exists(result_path):
            continue  # skip missing

        with open(result_path, "rb") as f:
            loaded_results = pickle.load(f)

        results = evaluate_by_batch(loaded_results, batch_size=batch_size)

        if target_attr not in results:
            continue  # skip if target attr not in result

        scores = results[target_attr]
        if batch_size is None:
            print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']}, Acc = {scores['Accuracy']}")
        else:
            print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']} ± {scores['F1-std']}, Acc = {scores['Accuracy']} ± {scores['Acc-std']}")


Comparing '256' F1-score on attribute 'Young' under different do(.) conditions:
do(Young)	Young: F1 = 0.591 ± 0.04, Acc = 0.661 ± 0.03
do(Male)	Young: F1 = 0.898 ± 0.02, Acc = 0.848 ± 0.02
do(No_Beard)	Young: F1 = 0.941 ± 0.01, Acc = 0.907 ± 0.02
do(Bald)	Young: F1 = 0.889 ± 0.02, Acc = 0.85 ± 0.02
Comparing '256' F1-score on attribute 'Male' under different do(.) conditions:
do(Young)	Male: F1 = 0.995 ± 0.0, Acc = 0.996 ± 0.0
do(Male)	Male: F1 = 0.999 ± 0.0, Acc = 0.999 ± 0.0
do(No_Beard)	Male: F1 = 0.779 ± 0.03, Acc = 0.76 ± 0.03
do(Bald)	Male: F1 = 0.937 ± 0.02, Acc = 0.943 ± 0.01
Comparing '256' F1-score on attribute 'No_Beard' under different do(.) conditions:
do(Young)	No_Beard: F1 = 0.996 ± 0.0, Acc = 0.994 ± 0.01
do(Male)	No_Beard: F1 = 0.965 ± 0.01, Acc = 0.968 ± 0.01
do(No_Beard)	No_Beard: F1 = 0.571 ± 0.05, Acc = 0.738 ± 0.03
do(Bald)	No_Beard: F1 = 0.977 ± 0.01, Acc = 0.961 ± 0.01
Comparing '256' F1-score on attribute 'Bald' under different do(.) conditions:
do(Young)	Bald:

In [2]:

for do_attr in do_folders:
    result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
    if not os.path.exists(result_path):
        continue  # skip missing

    with open(result_path, "rb") as f:
        loaded_results = pickle.load(f)
    print('do {} lipips distance is {}'.format(do_attr,np.mean(loaded_results['lpips'])))

do Young lipips distance is 0.05055588483810425
do Male lipips distance is 0.10380394011735916
do No_Beard lipips distance is 0.03226673603057861
do Bald lipips distance is 0.11150137335062027


In [ ]:
#dict_keys(['predictions', 'targets'])
loaded_results.keys()
#dict_keys(['Young', 'Male', 'No_Beard', 'Bald'])
loaded_results['predictions'].keys()

dict_keys(['predictions', 'targets'])

# FID and Minimality

In [1]:
import torch
minimality = torch.load("./saved/celeA_complex/FID/minimality.pt")
countefactual_images =  torch.load("./saved/celeA_complex/FID/counterfactual_tensors.pt")

/tmp/ipykernel_290807/4134258590.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  minimality = torch.load("./saved/celeA_complex/FID/minimality.pt")
/tmp/ipykernel_290807

In [6]:
minimality['factuals'][1].shape

AttributeError: 'tuple' object has no attribute 'shape'